# Week 06: RC Circuits -- Transient Response and Time Constants
## PHASE 2: Storing & Moving Charge

*Physics II (PHY102) . 3 Hours . Dr. Arif Solmaz*

## Learning Objectives

By the end of this week, you should be able to:

- Describe the charging and discharging behavior of an RC circuit
- Define the time constant and explain its physical meaning
- Write the exponential equations for voltage and current during charging and discharging
- Calculate the energy stored in a capacitor during charging
- Identify the 63.2% and 36.8% significance of the time constant
- Explain how an RC circuit acts as a low-pass filter
- Use semi-log plots to verify exponential behavior experimentally

## 🎯 Core Mastery Connection

RC circuits predict time behavior. $\tau = RC$ tells you how fast circuits charge and discharge. The exponential equations $V_C(t) = \varepsilon(1 - e^{-t/\tau})$ and $V_C(t) = V_0 e^{-t/\tau}$ follow directly from Kirchhoff's loop rule applied to a circuit with a capacitor -- the same framework you learned last week, now with time dependence.

> **Framework for every problem:** Configuration → Law → Equation → Prediction → Verify.

In [ ]:
# Setup
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Math
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox, Layout

plt.rcParams.update({'font.size': 13, 'figure.figsize': (9, 5)})
print('Week 06 setup complete.')

---
## 1. Review: Capacitors and DC Circuits

Recall from earlier weeks:
- A capacitor stores charge: $Q = CV$
- Energy stored: $U = \frac{1}{2}CV^2$

When we combine a capacitor with a resistor, the circuit behavior becomes **time-dependent**. The resistor limits how fast charge can flow onto or off of the capacitor plates.

---
## 2. RC Charging Circuit

When a battery of EMF $\varepsilon$ charges a capacitor through a resistor:

**Applying Kirchhoff's Loop Rule:**

$$\varepsilon - IR - \frac{Q}{C} = 0$$

Since $I = dQ/dt$, solving this differential equation gives:

$$V_C(t) = \varepsilon \left(1 - e^{-t/RC}\right)$$

$$I(t) = \frac{\varepsilon}{R} e^{-t/RC}$$

$$Q(t) = C\varepsilon \left(1 - e^{-t/RC}\right)$$

The **time constant** is:

$$\tau = RC$$

- At $t = \tau$: the capacitor has charged to **63.2%** of its final value
- At $t = 5\tau$: the capacitor is over **99%** charged (effectively fully charged)

### Interactive Demo 1: RC Charging Curve

Adjust R and C to see how the charging time constant changes.

In [ ]:
def rc_charging(R_kohm=1.0, C_uF=100.0, V_bat=10.0):
    R = R_kohm * 1e3  # Ohms
    C = C_uF * 1e-6   # Farads
    tau = R * C        # seconds
    
    t_max = 5 * tau
    t = np.linspace(0, t_max, 500)
    
    Vc = V_bat * (1 - np.exp(-t / tau))
    I = (V_bat / R) * np.exp(-t / tau)
    I_mA = I * 1000
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Voltage plot
    ax1.plot(t * 1000, Vc, 'b-', lw=2.5, label='$V_C(t)$')
    ax1.axhline(y=V_bat, color='gray', ls='--', lw=1, alpha=0.5, label=f'$\\varepsilon$ = {V_bat} V')
    ax1.axhline(y=0.632 * V_bat, color='red', ls=':', lw=1.5, alpha=0.7, label=f'63.2% = {0.632*V_bat:.2f} V')
    ax1.axvline(x=tau * 1000, color='red', ls=':', lw=1.5, alpha=0.7)
    ax1.plot(tau * 1000, 0.632 * V_bat, 'ro', ms=10, zorder=5)
    ax1.annotate(f'$\\tau$ = {tau*1000:.2f} ms', xy=(tau*1000, 0.632*V_bat),
                xytext=(tau*1000 + t_max*1000*0.15, 0.4*V_bat),
                fontsize=12, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Mark multiple tau
    for n in range(1, 6):
        pct = (1 - np.exp(-n)) * 100
        ax1.plot(n*tau*1000, V_bat*(1-np.exp(-n)), 'r.', ms=6)
        if n <= 3:
            ax1.text(n*tau*1000, V_bat*(1-np.exp(-n)) - 0.5,
                    f'{n}$\\tau$ ({pct:.1f}%)', fontsize=8, ha='center', color='red')
    
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Capacitor Voltage (V)')
    ax1.set_title('RC Charging: Voltage')
    ax1.legend(loc='right', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.5, V_bat * 1.15)
    
    # Current plot
    ax2.plot(t * 1000, I_mA, 'r-', lw=2.5, label='$I(t)$')
    I0_mA = V_bat / R * 1000
    ax2.axhline(y=I0_mA, color='gray', ls='--', lw=1, alpha=0.5, label=f'$I_0$ = {I0_mA:.3f} mA')
    ax2.axhline(y=0.368 * I0_mA, color='blue', ls=':', lw=1.5, alpha=0.7, label=f'36.8% = {0.368*I0_mA:.3f} mA')
    ax2.axvline(x=tau * 1000, color='blue', ls=':', lw=1.5, alpha=0.7)
    ax2.plot(tau * 1000, 0.368 * I0_mA, 'bo', ms=10, zorder=5)
    
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Current (mA)')
    ax2.set_title('RC Charging: Current')
    ax2.legend(loc='right', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'  R = {R_kohm} kΩ,  C = {C_uF} μF')
    print(f'  τ = RC = {R_kohm}×10³ × {C_uF}×10⁻⁶ = {tau*1000:.3f} ms')
    print(f'  Initial current: I₀ = ε/R = {V_bat}/{R:.0f} = {I0_mA:.3f} mA')
    print(f'  At t = τ: V_C = {0.632*V_bat:.3f} V (63.2%),  I = {0.368*I0_mA:.3f} mA (36.8%)')
    print(f'  Capacitor fully charged (~99%) at t ≈ 5τ = {5*tau*1000:.2f} ms')

interact(rc_charging,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)'),
         V_bat=FloatSlider(min=1, max=20, step=0.5, value=10, description='ε (V)'));

---
## 3. RC Discharging Circuit

When a fully charged capacitor discharges through a resistor (no battery):

$$V_C(t) = V_0 \, e^{-t/RC}$$

$$I(t) = -\frac{V_0}{R} \, e^{-t/RC}$$

The negative sign indicates current flows in the opposite direction during discharge.

- At $t = \tau$: voltage has dropped to **36.8%** of its initial value
- The capacitor loses 63.2% of its voltage in one time constant

### Interactive Demo 2: RC Discharging Curve

In [ ]:
def rc_discharging(R_kohm=1.0, C_uF=100.0, V0=10.0):
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    tau = R * C
    
    t_max = 5 * tau
    t = np.linspace(0, t_max, 500)
    
    Vc = V0 * np.exp(-t / tau)
    I = -(V0 / R) * np.exp(-t / tau)
    I_mA = I * 1000
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Voltage plot
    ax1.plot(t * 1000, Vc, 'b-', lw=2.5, label='$V_C(t) = V_0 e^{-t/\\tau}$')
    ax1.axhline(y=0.368 * V0, color='red', ls=':', lw=1.5, alpha=0.7, label=f'36.8% = {0.368*V0:.2f} V')
    ax1.axvline(x=tau * 1000, color='red', ls=':', lw=1.5, alpha=0.7)
    ax1.plot(tau * 1000, 0.368 * V0, 'ro', ms=10, zorder=5)
    ax1.annotate(f'$\\tau$ = {tau*1000:.2f} ms', xy=(tau*1000, 0.368*V0),
                xytext=(tau*1000 + t_max*1000*0.15, 0.6*V0),
                fontsize=12, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Shade area to show energy dissipated
    ax1.fill_between(t*1000, Vc, alpha=0.1, color='blue')
    
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Capacitor Voltage (V)')
    ax1.set_title('RC Discharging: Voltage (Exponential Decay)')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-0.5, V0 * 1.15)
    
    # Current plot
    ax2.plot(t * 1000, I_mA, 'r-', lw=2.5, label='$I(t)$ (discharge)')
    ax2.axhline(y=0, color='black', lw=0.5)
    I0_mA = -V0 / R * 1000
    ax2.axhline(y=0.368 * I0_mA, color='blue', ls=':', lw=1.5, alpha=0.7)
    ax2.axvline(x=tau * 1000, color='blue', ls=':', lw=1.5, alpha=0.7)
    ax2.plot(tau * 1000, 0.368 * I0_mA, 'bo', ms=10, zorder=5)
    
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Current (mA)')
    ax2.set_title('RC Discharging: Current (Negative = Reversed)')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'  τ = RC = {tau*1000:.3f} ms')
    print(f'  At t = τ: V_C = {0.368*V0:.3f} V (36.8% of V₀)')
    print(f'  At t = 5τ: V_C = {np.exp(-5)*V0:.4f} V (0.67% of V₀ -- effectively zero)')

interact(rc_discharging,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)'),
         V0=FloatSlider(min=1, max=20, step=0.5, value=10, description='V₀ (V)'));

---
## 4. Animation: RC Circuit Charging

Watch the capacitor charge in real time: the plates accumulate charge, the voltage rises, and the current decreases.

In [ ]:
fig_anim, axes_anim = plt.subplots(1, 3, figsize=(15, 5))

V_bat_anim = 10.0
R_anim = 1000  # Ohm
C_anim = 100e-6  # F
tau_anim = R_anim * C_anim  # 0.1 s
n_frames = 120
t_total = 5 * tau_anim

def animate_rc(frame):
    for ax in axes_anim:
        ax.clear()
    
    t_now = frame / n_frames * t_total
    Vc_now = V_bat_anim * (1 - np.exp(-t_now / tau_anim))
    I_now = (V_bat_anim / R_anim) * np.exp(-t_now / tau_anim)
    charge_frac = 1 - np.exp(-t_now / tau_anim)
    
    # Panel 1: Capacitor visualization
    ax1 = axes_anim[0]
    ax1.set_xlim(-2, 4)
    ax1.set_ylim(-1, 5)
    ax1.set_aspect('equal')
    ax1.axis('off')
    ax1.set_title('Capacitor', fontsize=13)
    
    # Plates
    plate_h = 3.0
    ax1.plot([0.8, 0.8], [0.5, 0.5 + plate_h], 'b-', lw=6)
    ax1.plot([1.8, 1.8], [0.5, 0.5 + plate_h], 'r-', lw=6)
    
    # Charges on plates (proportional to charge_frac)
    n_charges = int(charge_frac * 8)
    for i in range(n_charges):
        y_pos = 0.8 + i * (plate_h - 0.6) / 8
        ax1.text(0.5, y_pos, '+', fontsize=10, color='red', ha='center', fontweight='bold')
        ax1.text(2.1, y_pos, '−', fontsize=10, color='blue', ha='center', fontweight='bold')
    
    # Labels
    ax1.text(1.3, 4.2, f'V_C = {Vc_now:.2f} V', fontsize=12, ha='center', fontweight='bold', color='green')
    ax1.text(1.3, -0.3, f'Q = {charge_frac*100:.1f}%', fontsize=11, ha='center', color='purple')
    
    # Charge bar
    ax1.add_patch(patches.Rectangle((-1.5, 0.5), 0.6, plate_h * charge_frac,
                                     facecolor='orange', edgecolor='black', alpha=0.7))
    ax1.add_patch(patches.Rectangle((-1.5, 0.5), 0.6, plate_h,
                                     facecolor='none', edgecolor='black', lw=1.5))
    ax1.text(-1.2, 4.0, 'Charge', fontsize=9, ha='center')
    
    # Panel 2: Voltage vs time
    ax2 = axes_anim[1]
    t_arr = np.linspace(0, t_total, 300)
    Vc_arr = V_bat_anim * (1 - np.exp(-t_arr / tau_anim))
    ax2.plot(t_arr * 1000, Vc_arr, 'b-', lw=1.5, alpha=0.3)
    
    # Plot up to current time
    mask = t_arr <= t_now
    ax2.plot(t_arr[mask] * 1000, Vc_arr[mask], 'b-', lw=2.5)
    ax2.plot(t_now * 1000, Vc_now, 'ro', ms=8, zorder=5)
    
    ax2.axhline(y=V_bat_anim, color='gray', ls='--', lw=1, alpha=0.5)
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Voltage (V)')
    ax2.set_title(f'V_C(t)   [t = {t_now*1000:.1f} ms]')
    ax2.set_xlim(0, t_total * 1000)
    ax2.set_ylim(-0.5, V_bat_anim * 1.15)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Current vs time
    ax3 = axes_anim[2]
    I_arr = (V_bat_anim / R_anim) * np.exp(-t_arr / tau_anim) * 1000  # mA
    ax3.plot(t_arr * 1000, I_arr, 'r-', lw=1.5, alpha=0.3)
    ax3.plot(t_arr[mask] * 1000, I_arr[mask], 'r-', lw=2.5)
    ax3.plot(t_now * 1000, I_now * 1000, 'bo', ms=8, zorder=5)
    
    ax3.set_xlabel('Time (ms)')
    ax3.set_ylabel('Current (mA)')
    ax3.set_title(f'I(t)   [I = {I_now*1000:.3f} mA]')
    ax3.set_xlim(0, t_total * 1000)
    ax3.set_ylim(-0.5, V_bat_anim / R_anim * 1000 * 1.15)
    ax3.grid(True, alpha=0.3)
    
    fig_anim.tight_layout()

anim_rc = FuncAnimation(fig_anim, animate_rc, frames=n_frames, interval=60, blit=False)
plt.close(fig_anim)
HTML(anim_rc.to_jshtml())

---
## 5. The Time Constant in Detail

The time constant $\tau = RC$ has units of seconds:

$$[\tau] = [R][C] = \Omega \cdot F = \frac{V}{A} \cdot \frac{C}{V} = \frac{C}{A} = s$$

| Time | Charging $V_C / \varepsilon$ | Discharging $V_C / V_0$ |
|------|------|------|
| $t = \tau$ | 63.2% | 36.8% |
| $t = 2\tau$ | 86.5% | 13.5% |
| $t = 3\tau$ | 95.0% | 5.0% |
| $t = 4\tau$ | 98.2% | 1.8% |
| $t = 5\tau$ | 99.3% | 0.7% |

### Interactive Demo 3: Time Constant Explorer

See exactly where each multiple of tau falls on the charging and discharging curves.

In [ ]:
def time_constant_explorer(R_kohm=1.0, C_uF=100.0, V0=10.0):
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    tau = R * C
    
    t = np.linspace(0, 5 * tau, 500)
    Vc_charge = V0 * (1 - np.exp(-t / tau))
    Vc_discharge = V0 * np.exp(-t / tau)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Charging
    ax1.plot(t / tau, Vc_charge, 'b-', lw=2.5)
    ax1.axhline(y=V0, color='gray', ls='--', lw=1, alpha=0.5)
    
    colors_tau = ['red', 'orange', 'green', 'purple', 'brown']
    for n in range(1, 6):
        pct = (1 - np.exp(-n)) * 100
        val = V0 * (1 - np.exp(-n))
        ax1.plot(n, val, 'o', color=colors_tau[n-1], ms=10, zorder=5)
        ax1.axhline(y=val, color=colors_tau[n-1], ls=':', lw=1, alpha=0.4)
        ax1.axvline(x=n, color=colors_tau[n-1], ls=':', lw=1, alpha=0.4)
        ax1.text(n + 0.1, val - 0.4, f'{n}$\\tau$: {pct:.1f}%\n({val:.2f}V)',
                fontsize=9, color=colors_tau[n-1], fontweight='bold')
    
    ax1.set_xlabel('Time (multiples of $\\tau$)')
    ax1.set_ylabel('Capacitor Voltage (V)')
    ax1.set_title(f'Charging  ($\\tau$ = {tau*1000:.2f} ms)')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 5.5)
    
    # Discharging
    ax2.plot(t / tau, Vc_discharge, 'r-', lw=2.5)
    ax2.axhline(y=0, color='gray', ls='--', lw=1, alpha=0.5)
    
    for n in range(1, 6):
        pct = np.exp(-n) * 100
        val = V0 * np.exp(-n)
        ax2.plot(n, val, 'o', color=colors_tau[n-1], ms=10, zorder=5)
        ax2.axhline(y=val, color=colors_tau[n-1], ls=':', lw=1, alpha=0.4)
        ax2.axvline(x=n, color=colors_tau[n-1], ls=':', lw=1, alpha=0.4)
        ax2.text(n + 0.1, val + 0.2, f'{n}$\\tau$: {pct:.1f}%\n({val:.2f}V)',
                fontsize=9, color=colors_tau[n-1], fontweight='bold')
    
    ax2.set_xlabel('Time (multiples of $\\tau$)')
    ax2.set_ylabel('Capacitor Voltage (V)')
    ax2.set_title(f'Discharging  ($\\tau$ = {tau*1000:.2f} ms)')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 5.5)
    
    plt.tight_layout()
    plt.show()
    
    print(f'  τ = RC = {R_kohm} kΩ × {C_uF} μF = {tau*1000:.3f} ms')
    print(f'  After 1τ: charged to 63.2%, discharged to 36.8%')
    print(f'  After 5τ ({5*tau*1000:.1f} ms): circuit has essentially reached steady state')

interact(time_constant_explorer,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)'),
         V0=FloatSlider(min=1, max=20, step=0.5, value=10, description='V₀ (V)'));

---
## 6. Energy in RC Circuits

During charging, the battery delivers energy $U_{\text{battery}} = Q \varepsilon = C\varepsilon^2$.

The energy stored in the capacitor is:

$$U_C = \frac{1}{2}C\varepsilon^2$$

The remaining half is dissipated as heat in the resistor:

$$U_R = \frac{1}{2}C\varepsilon^2$$

Remarkably, exactly **half** the energy is always lost to the resistor during charging, regardless of the resistance value!

In [ ]:
def energy_rc(R_kohm=1.0, C_uF=100.0, V_bat=10.0):
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    tau = R * C
    
    t = np.linspace(0, 5*tau, 500)
    
    # Energy stored in capacitor
    Vc = V_bat * (1 - np.exp(-t/tau))
    U_cap = 0.5 * C * Vc**2 * 1000  # mJ
    
    # Energy delivered by battery
    Q = C * V_bat * (1 - np.exp(-t/tau))
    U_bat = Q * V_bat * 1000  # mJ
    
    # Energy dissipated in resistor
    U_res = U_bat - U_cap  # mJ
    
    U_final_cap = 0.5 * C * V_bat**2 * 1000  # mJ
    U_final_bat = C * V_bat**2 * 1000  # mJ
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(t*1000, U_bat, 'b-', lw=2, label='Battery delivered')
    ax1.plot(t*1000, U_cap, 'g-', lw=2, label='Stored in capacitor')
    ax1.plot(t*1000, U_res, 'r-', lw=2, label='Dissipated in resistor')
    ax1.fill_between(t*1000, U_cap, alpha=0.2, color='green')
    ax1.fill_between(t*1000, U_cap, U_bat, alpha=0.2, color='red')
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Energy (mJ)')
    ax1.set_title('Energy During RC Charging')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Final energy pie chart
    ax2.pie([U_final_cap, U_final_cap],
            labels=[f'Stored in C\n{U_final_cap:.3f} mJ', f'Lost in R\n{U_final_cap:.3f} mJ'],
            colors=['green', 'red'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11})
    ax2.set_title(f'Final Energy Split\n(Battery delivered: {U_final_bat:.3f} mJ)')
    
    plt.tight_layout()
    plt.show()
    
    print(f'  Energy delivered by battery: U_bat = Cε² = {U_final_bat:.4f} mJ')
    print(f'  Energy stored in capacitor:  U_C = ½Cε² = {U_final_cap:.4f} mJ (50%)')
    print(f'  Energy lost in resistor:     U_R = ½Cε² = {U_final_cap:.4f} mJ (50%)')
    print(f'  The 50/50 split is independent of R!')

interact(energy_rc,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)'),
         V_bat=FloatSlider(min=1, max=20, step=0.5, value=10, description='ε (V)'));

---
## 7. Semi-Log Plot: Verifying Exponential Behavior

For a discharging RC circuit: $V_C(t) = V_0 e^{-t/\tau}$

Taking the natural log of both sides:

$$\ln(V_C) = \ln(V_0) - \frac{t}{\tau}$$

This is a **linear equation** in $t$ with slope $= -1/\tau$.

If we plot $\ln(V_C)$ vs $t$ and get a straight line, this **confirms** the exponential model. The slope gives us $\tau$.

In [ ]:
def semilog_verification(R_kohm=1.0, C_uF=100.0, V0=10.0, noise_level=0.0):
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    tau = R * C
    
    # Simulated "experimental" data with optional noise
    np.random.seed(42)
    n_points = 30
    t_data = np.linspace(0.05 * tau, 4 * tau, n_points)
    V_ideal = V0 * np.exp(-t_data / tau)
    V_noisy = V_ideal + noise_level * np.random.randn(n_points) * 0.1 * V0
    V_noisy = np.maximum(V_noisy, 0.01)  # Avoid log of zero/negative
    
    # Linear fit to ln(V) vs t
    ln_V = np.log(V_noisy)
    coeffs = np.polyfit(t_data, ln_V, 1)
    slope = coeffs[0]
    intercept = coeffs[1]
    tau_measured = -1.0 / slope
    V0_measured = np.exp(intercept)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Regular plot
    t_smooth = np.linspace(0, 4*tau, 300)
    ax1.plot(t_smooth*1000, V0 * np.exp(-t_smooth/tau), 'b-', lw=1.5, alpha=0.5, label='Theory')
    ax1.plot(t_data*1000, V_noisy, 'ro', ms=5, label='Data')
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Voltage (V)')
    ax1.set_title('Discharge Data (Linear Scale)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Semi-log plot
    ax2.plot(t_data*1000, ln_V, 'ro', ms=5, label='ln(V) data')
    fit_line = slope * t_smooth + intercept
    ax2.plot(t_smooth*1000, slope * t_smooth + intercept, 'b-', lw=2,
             label=f'Linear fit: slope = {slope:.2f} /s')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('ln(V)')
    ax2.set_title('Semi-Log Plot (should be linear)')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    error_pct = abs(tau_measured - tau) / tau * 100
    print(f'  True τ = {tau*1000:.3f} ms')
    print(f'  From slope: τ_measured = -1/slope = -1/({slope:.4f}) = {tau_measured*1000:.3f} ms')
    print(f'  Error: {error_pct:.2f}%')
    print(f'  V₀ from intercept: exp({intercept:.4f}) = {V0_measured:.3f} V (true: {V0} V)')
    if noise_level == 0:
        print(f'  The ln(V) vs t plot is perfectly linear -- confirming exponential decay.')
    else:
        print(f'  Even with noise, the linear trend in the semi-log plot confirms exponential behavior.')

interact(semilog_verification,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=10, max=1000, step=10, value=100, description='C (μF)'),
         V0=FloatSlider(min=1, max=20, step=0.5, value=10, description='V₀ (V)'),
         noise_level=FloatSlider(min=0, max=1, step=0.1, value=0, description='Noise'));

---
## 8. RC Low-Pass Filter

An RC circuit can act as a **frequency filter**. When driven by an AC signal:

$$\frac{V_{out}}{V_{in}} = \frac{1}{\sqrt{1 + (\omega RC)^2}} = \frac{1}{\sqrt{1 + (f/f_c)^2}}$$

The **cutoff frequency** is:

$$f_c = \frac{1}{2\pi RC}$$

- Below $f_c$: signals pass through with little attenuation
- Above $f_c$: signals are attenuated (blocked)
- At $f_c$: the output is reduced to $1/\sqrt{2} \approx 0.707$ of the input (the "$-3$ dB point")

### Interactive Demo 4: Low-Pass Filter Bode Plot

In [ ]:
def lowpass_filter(R_kohm=1.0, C_uF=100.0):
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    tau = R * C
    fc = 1.0 / (2 * np.pi * tau)  # Hz
    
    f = np.logspace(-1, 5, 500)  # Hz
    omega = 2 * np.pi * f
    
    # Transfer function magnitude
    H = 1.0 / np.sqrt(1 + (omega * tau)**2)
    H_dB = 20 * np.log10(H)
    
    # Phase
    phase = -np.arctan(omega * tau) * 180 / np.pi
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    
    # Magnitude (Bode)
    ax1.semilogx(f, H_dB, 'b-', lw=2.5)
    ax1.axhline(y=-3, color='red', ls='--', lw=1.5, alpha=0.7, label='−3 dB')
    ax1.axvline(x=fc, color='red', ls='--', lw=1.5, alpha=0.7, label=f'$f_c$ = {fc:.2f} Hz')
    ax1.plot(fc, -3, 'ro', ms=10, zorder=5)
    ax1.annotate(f'$f_c$ = {fc:.2f} Hz\n(−3 dB)', xy=(fc, -3),
                xytext=(fc * 5, -10), fontsize=11, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))
    
    # Asymptotic behavior
    f_high = f[f > 3*fc]
    slope_line = -20 * np.log10(f_high / fc)
    ax1.semilogx(f_high, slope_line, 'g--', lw=1.5, alpha=0.5, label='−20 dB/decade slope')
    
    ax1.set_ylabel('Magnitude (dB)')
    ax1.set_title(f'Bode Plot: RC Low-Pass Filter (R={R_kohm}kΩ, C={C_uF}μF)')
    ax1.legend(loc='lower left', fontsize=10)
    ax1.grid(True, which='both', alpha=0.3)
    ax1.set_ylim(-40, 5)
    
    # Phase
    ax2.semilogx(f, phase, 'r-', lw=2.5)
    ax2.axhline(y=-45, color='blue', ls='--', lw=1, alpha=0.5)
    ax2.axvline(x=fc, color='red', ls='--', lw=1.5, alpha=0.7)
    ax2.plot(fc, -45, 'ro', ms=10, zorder=5)
    ax2.text(fc * 1.5, -50, f'−45° at $f_c$', fontsize=11, color='red', fontweight='bold')
    
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Phase (degrees)')
    ax2.set_title('Phase Response')
    ax2.grid(True, which='both', alpha=0.3)
    ax2.set_ylim(-95, 5)
    
    plt.tight_layout()
    plt.show()
    
    print(f'  Cutoff frequency: f_c = 1/(2π RC) = 1/(2π × {R:.0f} × {C*1e6:.0f}μF) = {fc:.3f} Hz')
    print(f'  At f_c: |H| = 1/√2 ≈ 0.707 (−3 dB), Phase = −45°')
    print(f'  For f >> f_c: gain drops at −20 dB/decade (−6 dB/octave)')

interact(lowpass_filter,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=1, max=1000, step=1, value=100, description='C (μF)'));

### Visualizing the Filter Effect on a Signal

In [ ]:
def filter_signal_demo(R_kohm=1.0, C_uF=100.0, f_signal=5.0):
    """Show input vs output of an RC low-pass filter for a sine wave."""
    R = R_kohm * 1e3
    C = C_uF * 1e-6
    fc = 1.0 / (2 * np.pi * R * C)
    
    omega = 2 * np.pi * f_signal
    tau = R * C
    
    # Gain and phase shift
    gain = 1.0 / np.sqrt(1 + (omega * tau)**2)
    phi = -np.arctan(omega * tau)
    
    t = np.linspace(0, 3.0 / max(f_signal, 0.1), 500)
    Vin = np.sin(omega * t)
    Vout = gain * np.sin(omega * t + phi)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(t * 1000, Vin, 'b-', lw=2, label='Input $V_{in}$', alpha=0.7)
    ax.plot(t * 1000, Vout, 'r-', lw=2.5, label=f'Output $V_{{out}}$ (gain={gain:.3f})')
    ax.axhline(y=gain, color='red', ls=':', lw=1, alpha=0.5)
    ax.axhline(y=-gain, color='red', ls=':', lw=1, alpha=0.5)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Voltage (normalized)')
    ax.set_title(f'Low-Pass Filter: f_signal = {f_signal:.1f} Hz, f_c = {fc:.2f} Hz, '
                 f'Gain = {gain:.3f} ({20*np.log10(gain):.1f} dB)')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Indicate whether signal passes or is blocked
    if f_signal < fc:
        status = 'PASSES (f < fc)'
        color = 'green'
    elif f_signal < 2*fc:
        status = 'ATTENUATED (f ~ fc)'
        color = 'orange'
    else:
        status = 'BLOCKED (f >> fc)'
        color = 'red'
    ax.text(0.02, 0.95, status, transform=ax.transAxes, fontsize=14,
            fontweight='bold', color=color, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

interact(filter_signal_demo,
         R_kohm=FloatSlider(min=0.1, max=10, step=0.1, value=1.0, description='R (kΩ)'),
         C_uF=FloatSlider(min=1, max=500, step=1, value=100, description='C (μF)'),
         f_signal=FloatSlider(min=0.1, max=50, step=0.1, value=5.0, description='f_signal (Hz)'));

---
## 9. Worked Examples

### Example 1: RC Time Constant Calculation

**Problem:** A 2.2 kΩ resistor is connected in series with a 47 μF capacitor and a 12V battery.
1. Calculate the time constant
2. Find the capacitor voltage and current at t = 50 ms
3. How long until the capacitor is 90% charged?

In [ ]:
# Worked Example 1
R = 2200   # Ohm
C = 47e-6  # F
V_bat = 12.0  # V

print('='*60)
print('WORKED EXAMPLE 1: RC Time Constant')
print('='*60)

# Part 1: Time constant
tau = R * C
print(f'\nPart 1: Time constant')
print(f'  τ = RC = {R} Ω × {C*1e6} μF = {tau*1000:.3f} ms')

# Part 2: Values at t = 50 ms
t = 50e-3  # 50 ms
Vc = V_bat * (1 - np.exp(-t / tau))
I = (V_bat / R) * np.exp(-t / tau)
print(f'\nPart 2: At t = {t*1000} ms (= {t/tau:.2f}τ)')
print(f'  V_C = ε(1 − e^(−t/τ)) = {V_bat}(1 − e^(−{t*1000:.0f}/{tau*1000:.3f}))')
print(f'  V_C = {V_bat}(1 − e^({-t/tau:.4f})) = {V_bat}(1 − {np.exp(-t/tau):.6f})')
print(f'  V_C = {Vc:.4f} V')
print(f'  I = (ε/R)e^(−t/τ) = ({V_bat}/{R})e^({-t/tau:.4f}) = {I*1000:.4f} mA')

# Part 3: Time to 90% charged
# V_C = 0.9ε => 1 - e^(-t/τ) = 0.9 => e^(-t/τ) = 0.1 => t = -τ ln(0.1)
t_90 = -tau * np.log(0.1)
print(f'\nPart 3: Time to 90% charged')
print(f'  0.9 = 1 − e^(−t/τ)  =>  e^(−t/τ) = 0.1  =>  t = −τ ln(0.1)')
print(f'  t = −{tau*1000:.3f} × ln(0.1) = −{tau*1000:.3f} × (−2.303)')
print(f'  t = {t_90*1000:.3f} ms = {t_90/tau:.3f}τ')

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
t_arr = np.linspace(0, 5*tau, 300)
Vc_arr = V_bat * (1 - np.exp(-t_arr/tau))
ax.plot(t_arr*1000, Vc_arr, 'b-', lw=2)
ax.axhline(y=V_bat, color='gray', ls='--', lw=1, alpha=0.5)
ax.plot(t*1000, Vc, 'ro', ms=10, zorder=5, label=f't = {t*1000:.0f} ms: V = {Vc:.2f} V')
ax.plot(t_90*1000, 0.9*V_bat, 'g^', ms=10, zorder=5, label=f't = {t_90*1000:.1f} ms: V = {0.9*V_bat:.1f} V (90%)')
ax.axhline(y=0.9*V_bat, color='green', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('V_C (V)')
ax.set_title(f'RC Charging: R={R}Ω, C={C*1e6}μF, τ={tau*1000:.1f}ms')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Example 2: Energy Stored During Charging

**Problem:** A 10 μF capacitor is charged through a 5 kΩ resistor from a 9V battery.
1. How much energy does the battery deliver in total?
2. How much energy is stored in the capacitor?
3. How much energy is dissipated in the resistor?
4. What is the maximum instantaneous power dissipated in the resistor?

In [ ]:
# Worked Example 2
R2 = 5000     # Ohm
C2 = 10e-6    # F
V2 = 9.0      # V
tau2 = R2 * C2

print('='*60)
print('WORKED EXAMPLE 2: Energy in RC Charging')
print('='*60)

# Part 1
U_bat = C2 * V2**2
print(f'\nPart 1: Energy delivered by battery')
print(f'  U_bat = Qε = (Cε)ε = Cε² = {C2*1e6} μF × ({V2})² = {U_bat*1e6:.1f} μJ')

# Part 2
U_cap = 0.5 * C2 * V2**2
print(f'\nPart 2: Energy stored in capacitor')
print(f'  U_C = ½Cε² = ½ × {C2*1e6} μF × ({V2})² = {U_cap*1e6:.1f} μJ')

# Part 3
U_res = U_bat - U_cap
print(f'\nPart 3: Energy dissipated in resistor')
print(f'  U_R = U_bat − U_C = {U_bat*1e6:.1f} − {U_cap*1e6:.1f} = {U_res*1e6:.1f} μJ')
print(f'  (Exactly half the battery energy is always lost to the resistor!)')

# Part 4
I_max = V2 / R2
P_max = I_max**2 * R2
print(f'\nPart 4: Maximum instantaneous power in resistor')
print(f'  I_max occurs at t = 0: I₀ = ε/R = {V2}/{R2} = {I_max*1000:.3f} mA')
print(f'  P_max = I₀²R = ({I_max*1000:.3f} mA)² × {R2} Ω = {P_max*1000:.3f} mW')
print(f'  Or equivalently: P_max = ε²/R = {V2}²/{R2} = {V2**2/R2*1000:.3f} mW')

# Power plot
fig, ax = plt.subplots(figsize=(9, 5))
t = np.linspace(0, 5*tau2, 300)
I_t = (V2/R2) * np.exp(-t/tau2)
P_t = I_t**2 * R2 * 1000  # mW
ax.plot(t*1000, P_t, 'r-', lw=2.5)
ax.fill_between(t*1000, P_t, alpha=0.3, color='red', label=f'Total area = U_R = {U_res*1e6:.1f} μJ')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Power (mW)')
ax.set_title('Instantaneous Power Dissipated in Resistor During Charging')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Multiple RC Time Constants Comparison

Different R and C values give different response speeds. Here we compare several RC combinations.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

configs = [
    (1e3, 100e-6, '1kΩ, 100μF'),
    (1e3, 47e-6,  '1kΩ, 47μF'),
    (2.2e3, 100e-6, '2.2kΩ, 100μF'),
    (470, 100e-6, '470Ω, 100μF'),
    (10e3, 10e-6, '10kΩ, 10μF'),
]

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
t = np.linspace(0, 1.5, 500)  # seconds

for (R, C, label), color in zip(configs, colors):
    tau = R * C
    Vc = 1 - np.exp(-t / tau)
    ax.plot(t * 1000, Vc * 100, lw=2, color=color,
            label=f'{label} (τ = {tau*1000:.0f} ms)')

ax.axhline(y=63.2, color='gray', ls='--', lw=1, alpha=0.5)
ax.text(1050, 64, '63.2%', fontsize=10, color='gray')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Charging Progress (%)')
ax.set_title('Comparing RC Charging Speeds')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1500)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

---
## 11. Summary

| Concept | Key Equation |
|---|---|
| Time constant | $\tau = RC$ |
| Charging voltage | $V_C(t) = \varepsilon(1 - e^{-t/\tau})$ |
| Charging current | $I(t) = (\varepsilon/R)\,e^{-t/\tau}$ |
| Discharging voltage | $V_C(t) = V_0\,e^{-t/\tau}$ |
| Energy stored | $U_C = \frac{1}{2}CV^2$ |
| At $t = \tau$ | Charged to 63.2%, discharged to 36.8% |
| Low-pass cutoff | $f_c = 1/(2\pi RC)$ |
| Semi-log slope | Slope of $\ln V$ vs $t$ gives $-1/\tau$ |

---
## Problem Set

> **For each problem: Identify the configuration → Choose the law → Write the equation → Predict → Verify**

Work through the following problems on RC circuits, time constants, charging/discharging, and filtering. Problems are graded by difficulty:
- **L1 (Basic):** Single-concept, direct application of formulas
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

---

### L1 -- Basic Problems

**P1.** A $1.0\;\text{k}\Omega$ resistor is connected in series with a $47\;\mu$F capacitor. Calculate the time constant $\tau$.

<details><summary>Answer</summary>47.0 ms</details>

In [ ]:
# ✏️ [P1] Your solution here


**P2.** An RC circuit with $R = 2.2\;\text{k}\Omega$ and $C = 22\;\mu$F is connected to a $5.0$ V battery at $t = 0$. Find the capacitor voltage at $t = 0.10$ s.

<details><summary>Answer</summary>4.52 V</details>

In [ ]:
# ✏️ [P2] Your solution here


**P3.** A capacitor initially charged to $V_0 = 8.0$ V discharges through a $4.7\;\text{k}\Omega$ resistor. The capacitance is $C = 10\;\mu$F. Find the voltage and the percentage of initial charge remaining at $t = 100$ ms.

<details><summary>Answer</summary>V(100 ms) = 0.968 V, 12.1% remaining</details>

In [ ]:
# ✏️ [P3] Your solution here


**P4.** How many time constants does it take for a charging RC circuit to reach 95% of the battery voltage?

<details><summary>Answer</summary>t = −τ ln(0.05) = 3.00τ</details>

In [ ]:
# ✏️ [P4] Your solution here


### L2 -- Intermediate Problems

**P5.** A $100\;\mu$F capacitor is fully charged to $12.0$ V and then discharged through a $1.4\;\text{k}\Omega$ resistor. Find (a) the initial discharge current, (b) the current at $t = 200$ ms, and (c) the total energy initially stored in the capacitor.

<details><summary>Answer</summary>(a) 8.57 mA, (b) 2.06 mA, (c) 7.20 mJ</details>

In [ ]:
# ✏️ [P5] Your solution here


**P6.** An RC circuit with $R = 3.3\;\text{k}\Omega$ and $C = 22\;\mu$F is being charged from a $6.0$ V battery. How long does it take for the capacitor voltage to reach exactly $4.0$ V?

<details><summary>Answer</summary>t = −τ ln(1 − 4/6) = −τ ln(1/3) = 79.8 ms</details>

In [ ]:
# ✏️ [P6] Your solution here


**P7.** An RC low-pass filter is built with $R = 10\;\text{k}\Omega$ and $C = 1.0\;\mu$F. (a) Calculate the cutoff frequency $f_c$. (b) What is the gain in dB at a frequency of $100$ Hz?

<details><summary>Answer</summary>(a) f_c = 15.9 Hz, (b) −16.2 dB</details>

In [ ]:
# ✏️ [P7] Your solution here


**P8.** A $15\;\mu$F capacitor is charged from a $30$ V battery through a $4.5\;\text{k}\Omega$ resistor. (a) How much total energy does the battery deliver? (b) How much energy is stored in the capacitor? (c) Verify that exactly half the energy is dissipated in the resistor.

<details><summary>Answer</summary>(a) U_bat = Cε² = 13.5 mJ, (b) U_C = ½Cε² = 6.75 mJ, (c) U_R = 6.75 mJ = 50%</details>

In [ ]:
# ✏️ [P8] Your solution here


### L3 -- Challenge Problems

**P9.** A camera flash circuit uses a $2500\;\mu$F capacitor charged through a $4.7\;\text{k}\Omega$ resistor from a $6.0$ V battery. The flash fires when the capacitor reaches $5.4$ V ($90\%$ of battery voltage). (a) How long must the user wait before the flash is ready? (b) When the flash fires, the capacitor discharges through the xenon tube (modeled as $R_{\text{tube}} = 0.10\;\Omega$). What is the peak current through the tube? (c) How much energy does the flash deliver?

<details><summary>Answer</summary>(a) t = −τ ln(0.1) = 27.1 s, (b) I_peak = 5.4/0.1 = 54 A, (c) U = ½CV² = 36.5 mJ (at 5.4 V)</details>

In [ ]:
# ✏️ [P9] Your solution here


**P10.** An ECG (electrocardiogram) amplifier needs a low-pass filter to reject $50$ Hz power-line noise while passing the ECG signal (up to $40$ Hz). (a) Design an RC filter with cutoff frequency $f_c = 40$ Hz using a $10\;\text{k}\Omega$ resistor and find the required capacitance. (b) What is the attenuation (in dB) of the $50$ Hz noise?

<details><summary>Answer</summary>(a) C = 1/(2πRf_c) = 0.398 μF, (b) At 50 Hz: |H| = 1/√(1+(50/40)²) = 0.625, gain = −4.08 dB</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## Bridge to Next Week

This week we studied how circuits respond to **sudden changes** (switching on/off a battery) -- the transient response of RC circuits.

Next week, we move from circuits to a new and fascinating topic: **Magnetism**. We will explore:
- **Magnetic fields** ($\vec{B}$) and how they differ from electric fields
- The **Lorentz force** on a moving charge: $\vec{F} = q\vec{v} \times \vec{B}$
- **Circular motion** of charged particles in uniform magnetic fields
- **Force on current-carrying wires**: $\vec{F} = I\vec{L} \times \vec{B}$
- Applications: mass spectrometers, velocity selectors, and the Hall effect

The interplay between electricity and magnetism is one of the great unifying themes of physics!